# Matching linearly-referenced road networks to Overture

This notebook is the runnable demo for matching a linearly-referenced road
dataset — the kind iRAP produces (Star Ratings and coded road attributes along
roads) — to Overture transportation, and producing a **bridge file** that links
the provider's internal IDs to Overture GERS IDs.

The prose companion at `9-transportation-matching.md` explains the methodology
and how to read the results; the `irap-integration-walkthrough.md` page frames
the integration decision. Read those alongside this notebook.

This notebook uses **real iRAP-methodology data**: the **AusRAP** (Australian
Road Assessment Program) Star Ratings for Victoria, published by the Victorian
Department of Transport and Planning as open data under **CC-BY 4.0**. It has
exactly the shape iRAP described — fixed 100 m per-carriageway segments, each
with a start/end GPS coordinate, a Vehicle Star Rating, and dozens of coded road
attributes — and it carries iRAP-style internal IDs (`Location ID`) with no GERS
IDs, so building the crosswalk to GERS is a genuine exercise, not a mock-up.

The demo area is **central Melbourne**, where AusRAP arterial coverage overlaps
Overture's dense road network. A compact Melbourne subset ships in
`data/ausrap_melbourne.csv`; the full state file and a bbox filter are documented
in the load step so you can re-scope to any Victorian area (or swap in a pilot
city / iRAP historical export later).

> **Attribution.** AusRAP data © State of Victoria (Department of Transport and
> Planning), licensed [CC-BY 4.0](https://creativecommons.org/licenses/by/4.0/).
> Source: https://opendata.transport.vic.gov.au/dataset/australian-road-assessment-program-ausrap

> **What's real vs. illustrative.** Every step runs on real data. Step 5 runs the
> `transportation-splitter`'s **actual split functions** (pure Python, no Spark) on
> real Overture segments — it needs an optional one-time install (see that step).
> The spatial match in step 6 is a compact stand-in for the full
> `transportation-matcher` — enough to produce a real bridge file, not the
> production matcher.

Steps:

1. Setup — imports, paths, current Overture release.
2. Area of interest — the Melbourne bbox to demo against.
3. Load Overture transportation (segments + connectors) for the AOI.
4. Load the real AusRAP (iRAP) network for the AOI.
5. Split Overture segments with the transportation-splitter's own functions.
6. Establish correspondence (spatial match) — real bridge from real data.
7. Assemble the bridge file.
8. Cardinality & two-rate diagnostic.


## 1. Setup

Imports, paths, and the current Overture release.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

# Resolve repo root from notebook location, so the notebook works whether
# Jupyter was launched from the repo root or from notebooks/.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA = REPO_ROOT / "data"
RESULTS = REPO_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

Overture publishes a STAC catalog of every release. The custom `latest` field
on the catalog points at the current release id, so we don't have to hardcode a
release that will go stale.

In [2]:
import pystac

catalog = pystac.Catalog.from_file("https://stac.overturemaps.org/catalog.json")
OVERTURE_RELEASE = catalog.extra_fields["latest"]
print(f"Using Overture release: {OVERTURE_RELEASE}")

Using Overture release: 2026-07-22.0


In [3]:
# DuckDB with spatial + httpfs so we can read Overture parquet from S3 directly.
con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")

## 2. Area of interest

Road matching is a local operation — pick a small, road-dense bbox to keep the
demo laptop-friendly. Replace this with the AOI that matches your iRAP sample.

The bbox is `(xmin, ymin, xmax, ymax)` in EPSG:4326.

In [4]:
# Central Melbourne, matching the shipped AusRAP subset. Widen/move this (and
# re-run the load step against the full state CSV) to demo a different area.
AOI = (144.90, -37.86, 145.02, -37.78)  # central Melbourne
xmin, ymin, xmax, ymax = AOI

## 3. Load Overture transportation for the AOI

We pull road `segment`s and their `connector`s for the bbox. Overture models
attributes (speed, surface, access, …) as **linearly-referenced** properties
scoped to `between` ranges along each segment — that linear referencing is
exactly what makes road matching a subline problem rather than a whole-feature
one. See lesson 9.

This step is runnable today.

In [5]:
THEME = "transportation"

seg_path = (
    f"s3://overturemaps-us-west-2/release/{OVERTURE_RELEASE}"
    f"/theme={THEME}/type=segment/*.parquet"
)

segments = con.execute(f"""
    SELECT
        id,
        subtype,
        class,
        names.primary AS name,
        ST_AsText(geometry) AS wkt
    FROM read_parquet('{seg_path}', filename=true, hive_partitioning=1)
    WHERE bbox.xmin <= {xmax} AND bbox.xmax >= {xmin}
      AND bbox.ymin <= {ymax} AND bbox.ymax >= {ymin}
      AND subtype = 'road'
""").fetchdf()

print(f"Loaded {len(segments):,} Overture road segments in AOI")
segments[["id", "class", "name"]].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 45,587 Overture road segments in AOI


,id,class,name
0,3f5729d2-3a17-42a8-bb6f-01d0e2781339,service,NaN
1,159cc38a-c914-458f-ae73-8b2ee1824b8c,cycleway,NaN
2,073118c3-ab9f-4324-a10e-a0964813b8fd,footway,NaN
3,e7c58b5b-4bf8-402a-8a65-41fbe2bbfb6f,residential,John Street
4,0f8e0310-f6d1-4f79-bca8-5bb79d743eec,residential,Mariner Street


In [6]:
con_path = (
    f"s3://overturemaps-us-west-2/release/{OVERTURE_RELEASE}"
    f"/theme={THEME}/type=connector/*.parquet"
)

connectors = con.execute(f"""
    SELECT id, ST_AsText(geometry) AS wkt
    FROM read_parquet('{con_path}', filename=true, hive_partitioning=1)
    WHERE bbox.xmin <= {xmax} AND bbox.xmax >= {xmin}
      AND bbox.ymin <= {ymax} AND bbox.ymax >= {ymin}
""").fetchdf()

print(f"Loaded {len(connectors):,} Overture connectors in AOI")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 76,655 Overture connectors in AOI


## 4. Load the real AusRAP (iRAP) network for the AOI

AusRAP data has exactly the shape iRAP described (see
`irap-integration-walkthrough.md`):

- **Fixed 100 m segmentation** — one row per 100 m, with `Latitude`/`Longitude`
  (start) and `End Latitude`/`End Longitude` (end). *Not* arbitrary LR.
- **Per-carriageway** — the `Carriageway` field; a divided road is captured as
  separate directional runs, so one Overture road line often corresponds to two
  AusRAP lines.
- **"Worst within 100 m"** — each bin's rating summarizes the worst hazard in
  that stretch; it's a bin-level value, not a point observation.
- **No GERS IDs** — the iRAP-style internal id here is `Location ID`; building
  the crosswalk to GERS is the whole job.
- **Dozens of coded attributes** (speed limit, roadside severity, lane width,
  delineation, pedestrian crossing, sidewalk, …); `Vehicle Star Rating Smoothed`
  is the headline calculated field.

Following the repo convention, `data/` is gitignored — data is fetched locally,
not committed. The cell below **self-provisions**: it uses a cached
`data/ausrap_melbourne.csv` if present, else the full state `data/ausrap.csv`,
else downloads the full state CSV once (~77 MB, CC-BY 4.0) and caches it. The AOI
bbox filter is applied either way, so changing `AOI` re-scopes automatically.

In [7]:
import urllib.request

AUSRAP_URL = (
    "https://opendata.transport.vic.gov.au/dataset/"
    "da201a82-372b-4f23-8e88-9819a6672bd0/resource/"
    "4148542e-6e7b-476f-bcb2-de94114bc40a/download/ausrap.csv"
)
AUSRAP_SUBSET = DATA / "ausrap_melbourne.csv"
AUSRAP_FULL = DATA / "ausrap.csv"

if AUSRAP_SUBSET.exists():
    source = AUSRAP_SUBSET
elif AUSRAP_FULL.exists():
    source = AUSRAP_FULL
else:
    DATA.mkdir(exist_ok=True)
    print("Downloading AusRAP Victoria (~77 MB, CC-BY 4.0, one-time)...")
    urllib.request.urlretrieve(AUSRAP_URL, AUSRAP_FULL)
    source = AUSRAP_FULL
print(f"AusRAP source: {source.name}")

AusRAP source: ausrap_melbourne.csv


In [8]:
# Build the provider frame from the real AusRAP fields. Each 100 m row becomes a
# LineString from its start to its end coordinate; Location ID is the iRAP-side
# identifier we will crosswalk to a GERS id.
provider = con.execute(f"""
    SELECT
        CAST("Location ID" AS VARCHAR)              AS provider_id,
        "Road name"                                  AS road_name,
        CAST("Carriageway" AS VARCHAR)               AS carriageway,
        "Vehicle Star Rating Smoothed"               AS star_rating,
        "Speed limit"                                AS speed_code,
        'LINESTRING(' || Longitude || ' ' || Latitude || ', '
                      || "End Longitude" || ' ' || "End Latitude" || ')' AS wkt
    FROM read_csv_auto('{source.as_posix()}', ignore_errors=true)
    WHERE Longitude BETWEEN {xmin} AND {xmax}
      AND Latitude  BETWEEN {ymin} AND {ymax}
      AND "End Longitude" IS NOT NULL AND "End Latitude" IS NOT NULL
""").fetchdf()

print(f"Loaded {len(provider):,} real AusRAP 100 m segments in AOI")
print("Star rating distribution:")
print(provider["star_rating"].value_counts().sort_index().to_string())
provider.head()

Loaded 2,284 real AusRAP 100 m segments in AOI
Star rating distribution:
star_rating
1      12
2     308
3    1397
4     520
5      47


,provider_id,road_name,carriageway,star_rating,speed_code,wkt
0,381547855,5762_MAIN HEIDELBERG-ELTHAM RD_REVERSE,2,3,7,"LINESTRING(145.0141135 -37.78397599, 145.01301..."
1,381547856,5762_MAIN HEIDELBERG-ELTHAM RD_REVERSE,2,3,7,"LINESTRING(145.0130127 -37.78417699, 145.01189..."
2,381547857,5762_MAIN HEIDELBERG-ELTHAM RD_REVERSE,2,3,7,"LINESTRING(145.0118926 -37.78431354, 145.01075..."
3,381547858,5762_MAIN HEIDELBERG-ELTHAM RD_REVERSE,2,3,7,"LINESTRING(145.010759 -37.78431023, 145.009638..."
4,381547859,5762_MAIN HEIDELBERG-ELTHAM RD_REVERSE,2,3,7,"LINESTRING(145.0096385 -37.78416641, 145.00852..."


## 5. Normalize segmentation with the transportation-splitter

Before matching, it helps to remove linear referencing as a variable: split each
Overture segment at its connectors (and `between` LR points) so every sub-segment
has exactly two connectors and no LRs — "the segment" and "the attribute range"
finally coincide.

![One LR-scoped Overture segment split into plain per-segment sub-segments](../img/transportation-splitter-object-model.svg)

The diagram shows the whole idea (see lesson 9): one segment carrying an attribute
on a `between [0.1, 0.6]` range becomes several sub-segments, each with two
connectors, no LRs, and a value that applies to its whole length. The splitter is
*not* a matcher — it doesn't establish correspondence or raise a match rate; it
makes an already-referenced network simpler to consume. That's why the split
output below is a **demonstration** and isn't fed into the step-6 match.

[`transportation-splitter`](https://github.com/OvertureMaps/transportation-splitter)
is a PySpark tool that does this at planet scale, but its **core split logic is
pure Python + Shapely** (`get_connector_split_points`, `add_lr_split_points`,
`split_line`). We call those functions directly below — the tool's *actual code*,
with **no Spark session and no Java** — to split this AOI's Overture segments.

> **Optional dependency.** The package lists `pyspark`/`apache-sedona`, so it's a
> declared optional extra, kept out of the default install to keep the base
> workshop light. If the import below fails, enable it — then re-run the cell (no
> Spark is ever started): `uv sync --extra splitter` (or `pip install ".[splitter]"`).
> At planet scale you'd run the full `OvertureTransportationSplitter` Spark
> pipeline instead.

In [9]:
try:
    from transportation_splitter import (
        get_connector_split_points, add_lr_split_points, split_line, get_length,
    )
    from transportation_splitter.models import JoinedConnector
    SPLITTER_AVAILABLE = True
    print("transportation-splitter available — using its real split functions.")
except ImportError:
    SPLITTER_AVAILABLE = False
    print("transportation-splitter not installed; skipping the live split.")
    print('Enable it with:  uv sync --extra splitter   (or  pip install ".[splitter]")')

transportation-splitter available — using its real split functions.


In [10]:
if SPLITTER_AVAILABLE:
    from shapely import wkt as shapely_wkt

    # Point lookup for the AOI connectors loaded in step 3.
    conn_pt = {cid: shapely_wkt.loads(w)
               for cid, w in zip(connectors["id"], connectors["wkt"])}

    SPLIT_SAMPLE = 4000  # keep the demo snappy; the operation is identical at scale
    seg_rows = con.execute(f"""
        SELECT id, ST_AsText(geometry) AS wkt, connectors
        FROM read_parquet('{seg_path}', filename=true, hive_partitioning=1)
        WHERE bbox.xmin <= {xmax} AND bbox.xmax >= {xmin}
          AND bbox.ymin <= {ymax} AND bbox.ymax >= {ymin}
          AND subtype = 'road'
        LIMIT {SPLIT_SAMPLE}
    """).fetchall()

    before = after = 0
    examples = []
    for sid, swkt, clist in seg_rows:
        geom = shapely_wkt.loads(swkt)
        if geom.geom_type != "LineString":
            continue
        length = get_length(geom)
        ordered = sorted(clist or [], key=lambda c: c["at"])
        joined = [JoinedConnector(c["connector_id"], conn_pt[c["connector_id"]], i, float(c["at"]))
                  for i, c in enumerate(ordered) if c["connector_id"] in conn_pt]
        points = get_connector_split_points(joined, geom, length)
        add_lr_split_points(points, [], sid, geom, length, 7, 0.01)  # [] = connectors only
        points = sorted(points, key=lambda p: p.lr)
        if len(points) < 2:
            continue
        subs = split_line(geom, points)
        before += 1
        after += len(subs)
        if len(subs) > 2 and len(examples) < 5:
            examples.append((sid[:12], len(subs)))

    print(f"Split {before:,} Overture segments -> {after:,} sub-segments "
          f"(avg {after / max(before, 1):.2f} per segment) using the splitter's own code.")
    print("Each sub-segment now has exactly two connectors and no linear references —")
    print("the simple form the matcher and downstream consumers want (lesson 9).")
    if examples:
        print("Examples (segment_id, sub-segments):", examples)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Split 3,951 Overture segments -> 11,140 sub-segments (avg 2.82 per segment) using the splitter's own code.
Each sub-segment now has exactly two connectors and no linear references —
the simple form the matcher and downstream consumers want (lesson 9).
Examples (segment_id, sub-segments): [('0dff6225-614', 4), ('091e2096-115', 3), ('a10e1493-408', 6), ('6f366b9f-7af', 3), ('afe629f7-b05', 4)]


**Why this matters here (lesson 9):** splitting removes linear referencing as a
variable before matching, and — once a bridge exists — the same tool ingests the
iRAP feed directly: join the iRAP attributes onto Overture `id`s, then split once
to get analysis-ready per-segment data. The functions above are exactly the logic
the planet-scale Spark pipeline runs, just called directly on a sample here.

> **What the split does — and doesn't — change downstream.** This step is a
> standalone demonstration: it reports the segmentation fan-out, but the compact
> spatial matcher in step 6 deliberately runs on the **original** Overture
> segments. Enabling or skipping the splitter therefore does **not** change the
> bridge file or the two-rate diagnostic — that's expected, not a bug. The
> splitter is *not* a matcher: it adds no correspondence logic and barely moves
> the match rate. What it changes is Overture's segmentation *granularity*, which
> reshapes the **cardinality mix** — and not in the obvious direction: the
> optional **§9** below actually runs the experiment and finds the clean rate
> shifts slightly *up* on this AOI (finer segments sit closer to the 100 m bins),
> not down. The splitter's real payoff is the **last-mile ingest** case described
> above — explode a bridged, LR-scoped iRAP feed into plain per-segment rows —
> plus normalizing segmentation when you need to compare LR-scoped attributes
> directly. Neither of those is what step 6 is doing.

## 6. Establish correspondence (spatial match)

[`transportation-matcher`](https://github.com/OvertureMaps/transportation-matcher)
matches linearly-referenced road networks with differing segmentations via a
five-step pipeline (spatial-join → LR sublines → score → topology update →
infer). The full tool computes true subline LR ranges; here we run a compact,
**spatial-only** stand-in — the same shape as the HPMS precedent (a buffered
match, no shared IDs) — which is enough to produce a *real* bridge from the real
AusRAP and Overture geometry. It is not the production matcher.

> This match runs on the **original** Overture segments from step 3, *not* on
> step 5's split output — see the note at the end of step 5 for why the splitter
> isn't in this path and why that leaves the numbers below unchanged.

We buffer each Overture segment by a tight tolerance (~15 m, as in the HPMS
precedent) and measure the fraction of each 100 m AusRAP bin that falls within it
— keeping only segments that cover at least half the bin. Measuring bin *coverage*
(rather than raw proximity) is what stops a fat buffer from spuriously linking
parallel and cross streets in a dense grid. Against Overture's finer,
intersection-split segmentation the messy buckets (merge / split / mixed)
dominate and **clean** is rare — exactly the linear-cardinality story from lesson
9, now on real data.

In [11]:
con.register("provider", provider)
con.register("segments", segments[["id", "wkt"]])

BUFFER_DEG = 0.00015   # ~15 m match tolerance (as in HPMS); data is EPSG:4326
COVER_MIN = 0.5        # a segment must cover >= 50% of the 100 m bin to count

candidates = con.execute(f"""
    WITH pairs AS (
      SELECT p.provider_id, s.id AS gers_id,
             ST_Length(ST_Intersection(
                 ST_Buffer(ST_GeomFromText(s.wkt), {BUFFER_DEG}),
                 ST_GeomFromText(p.wkt))) AS covered,
             ST_Length(ST_GeomFromText(p.wkt)) AS bin_len
      FROM provider p
      JOIN segments s
        ON ST_Intersects(ST_Buffer(ST_GeomFromText(s.wkt), {BUFFER_DEG}),
                         ST_GeomFromText(p.wkt))
    )
    SELECT provider_id, gers_id, covered / NULLIF(bin_len, 0) AS cover
    FROM pairs
""").fetchdf()
candidates = candidates[candidates["cover"] >= COVER_MIN]

print(f"{len(candidates):,} candidate pairs (segment covers >= {COVER_MIN:.0%} of bin)")
candidates.sort_values("cover", ascending=False).head()

6,059 candidate pairs (segment covers >= 50% of bin)


,provider_id,gers_id,cover
11545,381558877,a9f60b10-bbcb-4201-9cf1-6d17c1bed108,1.0
12987,381563934,d867d98c-e030-4772-95e2-cc964def1b91,1.0
12875,381553554,7c4cceb7-acde-4763-87fb-9d5645b1e0c9,1.0
12884,381563102,83a8ec40-a493-4657-be91-7d1333dca368,1.0
12887,381563101,0aacc87c-02f0-4756-926d-ef2ec0e1ef86,1.0


## 7. Assemble the bridge file

The deliverable is the crosswalk between iRAP `Location ID`s and Overture GERS
IDs. We take each bin's best (largest-overlap) Overture match, then label the
observed cardinality (lesson 9):

- **clean** — the bin's best segment is matched by only this bin, and the bin
  matches only that segment (1:1).
- **merge** — several bins share one Overture segment (the common case for 100 m
  bins on longer segments).
- **split** — one bin overlaps several Overture segments (it straddles a
  boundary).
- **mixed** — both, and **unmatched** — no Overture overlap cleared the buffer.

`score` is the normalized overlap. True subline LR ranges (`provider_lr` /
`gers_lr`) require the full matcher; we record the provider side as the whole bin
`[0.0, 1.0]` and leave the GERS-side range for the matcher.

In [12]:
# Best (highest-coverage) Overture match per AusRAP bin.
best = (candidates.sort_values("cover", ascending=False)
                  .drop_duplicates("provider_id")
                  .set_index("provider_id"))

bins_per_gers = best.groupby("gers_id").size()             # merge indicator
segs_per_bin = candidates.groupby("provider_id")["gers_id"].nunique()  # split indicator

def classify(pid, gers_id):
    multi_seg = segs_per_bin.get(pid, 0) > 1
    multi_bin = bins_per_gers.get(gers_id, 0) > 1
    if multi_seg and multi_bin: return "mixed"
    if multi_seg: return "split"
    if multi_bin: return "merge"
    return "clean"

rows = []
for pid in provider["provider_id"]:
    if pid in best.index:
        g = best.at[pid, "gers_id"]
        rows.append((pid, g, "[0.0, 1.0]", "", classify(pid, g),
                     round(float(best.at[pid, "cover"]), 3)))
    else:
        rows.append((pid, None, "[0.0, 1.0]", "", "unmatched", 0.0))

bridge = pd.DataFrame(rows, columns=[
    "provider_id", "gers_id", "provider_lr", "gers_lr", "match_type", "score"])

out = RESULTS / "ausrap_overture_bridge.csv"
bridge.to_csv(out, index=False)
print(f"Wrote real bridge file ({len(bridge):,} rows): {out}")
print(bridge["match_type"].value_counts().to_string())
bridge.head()

Wrote real bridge file (2,284 rows): /Users/danabauer/overturemaps/workshop/results/ausrap_overture_bridge.csv
match_type
mixed        1219
split         776
merge         157
unmatched      99
clean          33


,provider_id,gers_id,provider_lr,gers_lr,match_type,score
0,381547855,9a1cdf7e-4c69-4e7b-90df-52695835dd08,"[0.0, 1.0]",,split,1.0
1,381547856,42677d95-422b-4522-a722-09aa48b9a86a,"[0.0, 1.0]",,mixed,1.0
2,381547857,42677d95-422b-4522-a722-09aa48b9a86a,"[0.0, 1.0]",,mixed,1.0
3,381547858,42677d95-422b-4522-a722-09aa48b9a86a,"[0.0, 1.0]",,mixed,1.0
4,381547859,d38205ce-db7c-43bd-96bc-4c978a4cecbb,"[0.0, 1.0]",,split,1.0


## 8. Cardinality & two-rate diagnostic

Lesson 8's framing, in linear form: a **match rate** (fraction of AusRAP bins
with any Overture correspondence) and a **clean rate** (of matched bins, fraction
that are 1:1). The combination — not either number alone — tells you whether
you're seeing a coverage gap, a segmentation-model difference, or a clean-adoption
case. These now run on the real match above.

In [13]:
n_provider = provider["provider_id"].nunique()
matched = bridge[bridge["gers_id"].notna()]
clean = bridge[bridge["match_type"] == "clean"]

match_rate = len(matched) / n_provider if n_provider else 0.0
clean_rate = len(clean) / len(matched) if len(matched) else 0.0

print(f"AusRAP bins in AOI: {n_provider:,}")
print(f"Match rate: {match_rate:6.1%}  ({len(matched):,} matched)")
print(f"Clean rate: {clean_rate:6.1%}  ({len(clean):,} clean of matched)")
print()
print("Reading it (lesson 9): high match + very low clean is the signature of a "
      "segmentation-model difference, not a failure. 100 m bins against "
      "Overture's finer, intersection-split segments land as split/merge/mixed, "
      "so the right artifact is an LR-ranged link table, not direct GERS "
      "adoption. The unmatched bucket is not an audit result.")

AusRAP bins in AOI: 2,284
Match rate:  95.7%  (2,185 matched)
Clean rate:   1.5%  (33 clean of matched)

Reading it (lesson 9): high match + very low clean is the signature of a segmentation-model difference, not a failure. 100 m bins against Overture's finer, intersection-split segments land as split/merge/mixed, so the right artifact is an LR-ranged link table, not direct GERS adoption. The unmatched bucket is not an audit result.


## 9. (Optional) What changes if we match against the *split* network?

Steps 6–8 matched the AusRAP bins against Overture's **original** segments, and
step 5 ran the splitter only as a demonstration (its output was never used). This
optional section closes that loop: it splits the **whole** AOI, re-runs the *same*
spatial match against the split network, and puts the two diagnostics side by
side — so the effect of segmentation on matching is something you *measure*, not
something you take on faith.

What to keep in mind while reading it:

- **The splitter is not a matcher.** These cells add no new correspondence logic;
  they only swap a finer Overture segmentation into the exact same match. Any
  change in the numbers is caused purely by segmentation granularity.
- Overture's original segments span several connectors; the splitter cuts them at
  every connector, so the split network is **finer** (more, shorter segments).
  Fixed 100 m bins land differently on a finer network — the two-rate diagnostic
  makes the shift explicit.
- This needs the optional splitter dependency (same as step 5). Without it, the
  cells below no-op with an install hint.

This is lesson 9's core claim as a runnable experiment: change the segmentation,
change the *cardinality structure* of the match — which is why the honest
deliverable is an LR-ranged bridge file, not a single headline "match rate."

In [14]:
# --- Build the SPLIT Overture network for the ENTIRE AOI ---------------------
# Step 5 split only a 4,000-segment sample and threw the geometry away. Here we
# split *every* AOI road segment and KEEP each sub-segment's geometry, so the
# split network can be dropped into the match in place of the original segments.
# Same splitter functions as step 5 — still pure Python / Shapely, still no Spark.
if not SPLITTER_AVAILABLE:
    split_segments = None
    print("Splitter not installed — skipping the split-network comparison.")
    print('Enable it with:  uv sync --extra splitter   (or  pip install ".[splitter]")')
else:
    from shapely import wkt as shapely_wkt

    # Connector-id -> point lookup for the AOI (connectors loaded in step 3).
    # Built here so this section stands on its own, independent of step 5.
    conn_pt = {cid: shapely_wkt.loads(w)
               for cid, w in zip(connectors["id"], connectors["wkt"])}

    # Every AOI road segment WITH its connector list (no LIMIT this time).
    seg_rows_full = con.execute(f"""
        SELECT id, ST_AsText(geometry) AS wkt, connectors
        FROM read_parquet('{seg_path}', filename=true, hive_partitioning=1)
        WHERE bbox.xmin <= {xmax} AND bbox.xmax >= {xmin}
          AND bbox.ymin <= {ymax} AND bbox.ymax >= {ymin}
          AND subtype = 'road'
    """).fetchall()

    split_records = []
    for sid, swkt, clist in seg_rows_full:
        geom = shapely_wkt.loads(swkt)
        if geom.geom_type != "LineString":
            continue
        length = get_length(geom)
        # Order the segment's connectors by position (`at` is a 0..1 fraction),
        # keeping only those whose point we loaded for the AOI (step 3).
        ordered = sorted(clist or [], key=lambda c: c["at"])
        joined = [
            JoinedConnector(c["connector_id"], conn_pt[c["connector_id"]], i, float(c["at"]))
            for i, c in enumerate(ordered) if c["connector_id"] in conn_pt
        ]
        # Split at connectors only (we pass [] for extra LR points), which is the
        # dominant, routing-relevant effect. `split_line` returns SplitSegment
        # objects carrying the sub-geometry and its LR range on the parent.
        points = get_connector_split_points(joined, geom, length)
        add_lr_split_points(points, [], sid, geom, length, 7, 0.01)
        points = sorted(points, key=lambda p: p.lr)
        if len(points) < 2:
            split_records.append((f"{sid}@0.000-1.000", swkt))  # no interior connector
            continue
        for sub in split_line(geom, points):
            a, b = sub.start_split_point.lr, sub.end_split_point.lr
            # Sub-segment id = parent GERS id + its LR range: stable and readable.
            split_records.append((f"{sid}@{a:.3f}-{b:.3f}", sub.geometry.wkt))

    split_segments = pd.DataFrame(split_records, columns=["id", "wkt"])
    print(f"Original AOI road segments : {len(seg_rows_full):,}")
    print(f"Split sub-segments         : {len(split_segments):,} "
          f"(avg {len(split_segments) / max(len(seg_rows_full), 1):.2f} per segment)")

Original AOI road segments : 45,587
Split sub-segments         : 112,048 (avg 2.46 per segment)


### Re-run the match against both networks

To compare fairly, we apply the **identical** step 6–8 pipeline — spatial match
(15 m buffer, ≥ 50 % bin coverage) → best match per bin → cardinality
classification → two-rate diagnostic — to each network in turn. The helper below
is just steps 6–8 wrapped in a function, so the *only* thing that differs between
the two runs is the Overture-side segmentation.

(The split network has ~3× the rows, so its spatial join takes noticeably longer
than the original — give it a moment.)

In [15]:
# --- The step 6-8 pipeline as a reusable function ----------------------------
# Nothing here is new logic. It is exactly the spatial match (step 6), the
# best-match-per-bin bridge with cardinality labels (step 7), and the match-rate /
# clean-rate diagnostic (step 8) — parameterized on the Overture-side segment set
# so the SAME pipeline can run against either the ORIGINAL or the SPLIT network.
# That isolation is the point: the only thing differing between the two runs is
# Overture's segmentation, so any change in the numbers is attributable to it.
def match_and_diagnose(seg_df, prov_df, buffer_deg=BUFFER_DEG, cover_min=COVER_MIN):
    con.register("seg_cmp", seg_df[["id", "wkt"]])
    con.register("prov_cmp", prov_df)

    cand = con.execute(f"""
        WITH pairs AS (
          SELECT p.provider_id, s.id AS gers_id,
                 ST_Length(ST_Intersection(
                     ST_Buffer(ST_GeomFromText(s.wkt), {buffer_deg}),
                     ST_GeomFromText(p.wkt))) AS covered,
                 ST_Length(ST_GeomFromText(p.wkt)) AS bin_len
          FROM prov_cmp p
          JOIN seg_cmp s
            ON ST_Intersects(ST_Buffer(ST_GeomFromText(s.wkt), {buffer_deg}),
                             ST_GeomFromText(p.wkt))
        )
        SELECT provider_id, gers_id, covered / NULLIF(bin_len, 0) AS cover FROM pairs
    """).fetchdf()
    cand = cand[cand["cover"] >= cover_min]

    best = (cand.sort_values("cover", ascending=False)
                .drop_duplicates("provider_id").set_index("provider_id"))
    bins_per_gers = best.groupby("gers_id").size()                   # merge indicator
    segs_per_bin = cand.groupby("provider_id")["gers_id"].nunique()  # split indicator

    def classify(pid, gid):
        multi_seg = segs_per_bin.get(pid, 0) > 1
        multi_bin = bins_per_gers.get(gid, 0) > 1
        if multi_seg and multi_bin: return "mixed"
        if multi_seg: return "split"
        if multi_bin: return "merge"
        return "clean"

    labels = [classify(pid, best.at[pid, "gers_id"]) if pid in best.index else "unmatched"
              for pid in prov_df["provider_id"]]
    counts = pd.Series(labels).value_counts().to_dict()

    n = prov_df["provider_id"].nunique()
    matched = sum(v for k, v in counts.items() if k != "unmatched")
    clean = counts.get("clean", 0)
    return {
        "segments": len(seg_df),
        "match_rate": matched / n if n else 0.0,
        "clean_rate": clean / matched if matched else 0.0,
        "counts": counts,
    }

if not SPLITTER_AVAILABLE or globals().get("split_segments") is None:
    print("Split network unavailable — run the previous cell with the splitter installed.")
else:
    orig = match_and_diagnose(segments, provider)        # original Overture segmentation
    split = match_and_diagnose(split_segments, provider)  # splitter's finer segmentation

    # --- Side-by-side comparison ---------------------------------------------
    print(f"{'':20}{'ORIGINAL':>13}{'SPLIT':>13}")
    print(f"{'Overture segments':20}{orig['segments']:>13,}{split['segments']:>13,}")
    print(f"{'Match rate':20}{orig['match_rate']:>12.1%}{split['match_rate']:>13.1%}")
    print(f"{'Clean rate':20}{orig['clean_rate']:>12.1%}{split['clean_rate']:>13.1%}")
    print(f"\n{'cardinality':20}{'ORIGINAL':>13}{'SPLIT':>13}")
    for t in ["clean", "merge", "split", "mixed", "unmatched"]:
        print(f"{t:20}{orig['counts'].get(t, 0):>13,}{split['counts'].get(t, 0):>13,}")

    # Dynamic takeaway, so the notebook narrates whatever the data actually shows.
    delta = split["clean_rate"] - orig["clean_rate"]
    if abs(delta) < 0.01:
        verdict = "barely moves"
    elif delta > 0:
        verdict = f"RISES by {delta:.1%} (finer segments sit closer to the 100 m bin size)"
    else:
        verdict = f"FALLS by {-delta:.1%} (finer segments straddle more bin boundaries)"
    print(f"\nTakeaway: match rate stays ~unchanged; the clean rate {verdict}.")
    print("Either way it is the *cardinality mix* that shifts — the splitter is not a matcher.")

                         ORIGINAL        SPLIT
Overture segments          45,587      112,048
Match rate                 95.7%        94.4%
Clean rate                  1.5%         3.4%

cardinality              ORIGINAL        SPLIT
clean                          33           74
merge                         157          164
split                         776        1,295
mixed                       1,219          623
unmatched                      99          128

Takeaway: match rate stays ~unchanged; the clean rate RISES by 1.9% (finer segments sit closer to the 100 m bin size).
Either way it is the *cardinality mix* that shifts — the splitter is not a matcher.


### How to read the comparison

The **match rate barely moves** — both networks cover the same roads, so almost
every bin still finds *some* Overture correspondence. What changes is the
**cardinality mix**, and with it the clean rate:

- On the **original** network, an Overture segment is long: it both hosts many
  consecutive 100 m bins *and* gets straddled by them, so the biggest bucket is
  **mixed** and clean is rare (~1–2%).
- On the **split** network, segments are cut at every connector and so are much
  shorter. Each hosts far fewer bins, so the "many bins on one segment" condition
  weakens: most former **mixed** bins become plain **split** (a bin still straddles
  two shorter pieces), and a few become **clean** (a bin now lines up 1:1 with one
  short piece). Clean roughly doubles — but is still small in absolute terms, and
  **split** becomes the dominant bucket.

So on this AOI the clean rate goes *up* a little, which is the opposite of the
naive guess that "more boundaries = messier." The direction isn't the lesson,
though. The lesson is that **segmentation granularity is a knob that reshapes the
correspondence structure; it is not a matcher**. Whichever way the clean rate
moved, clean is still the small minority, so the honest deliverable is the same:
an **LR-ranged bridge file** that records each correspondence and its type, rather
than a single number that hides the split/merge reality. The splitter's real job
comes *after* such a bridge exists — normalizing the network for consumption —
which is exactly why the main pipeline (steps 6–8) does not route through it.